# Milestone 3: an Arabic transformer, tracked with MLflow

1. Runtime → Change runtime type → T4 GPU → Save.
2. Put your copy's address in `REPO` in the first code cell, then run the cells in order (Runtime → Run all works too).

The notebook makes two runs of CAMeLBERT and logs both in MLflow: the model scored as it is (a second baseline), then fine-tuned on your training reviews. It uses only the training and validation parts of `data/split.json`: the test part waits for milestone 5.

At the end, take a screenshot of the comparison table and save it as `docs/mlflow.png`, and download `results/transformer.json` from the Files panel (the folder icon on the left). Colab works on its own copy, so add both files to your copy (on GitHub: **Add file** → **Upload files**, into the right folders).

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/<your-username>/ai-capstone"  # Colab only: your copy's address

if "google.colab" in sys.modules and not os.path.exists("check.py"):
    if "<your-username>" in REPO:
        raise ValueError("Put your copy's address in REPO first, for example https://github.com/amina-s/ai-capstone")
    if not os.path.isdir("ai-capstone"):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir("ai-capstone")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # VS Code and Jupyter start in notebooks/
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)
print("Working in", os.getcwd())

In [ ]:
# Colab already has torch and transformers; this adds the rest. On your own computer, transformers also needs torch (a large download).
%pip install -q "datasets>=3.0" "transformers>=4.44" accelerate mlflow scikit-learn pandas

## The reviews

The same reviews, split and labels as milestones 1 and 2. LABR's `label` is the star rating (0 = 1 star … 4 = 5 stars): 1–2 stars are negative (0), 4–5 stars positive (1), and 3-star reviews are dropped. ar_res_reviews has `polarity` instead (0 negative, 1 positive).

In [ ]:
import json

import pandas as pd

if not os.path.exists("data/reviews.csv"):
    subprocess.run([sys.executable, "fetch_data.py"], check=True)  # the same reviews and split as milestone 1

split = json.load(open("data/split.json", encoding="utf-8"))
reviews = pd.read_csv("data/reviews.csv")
if "polarity" in reviews.columns:  # ar_res_reviews
    reviews["y"] = reviews["polarity"]
else:  # LABR: 1–2 stars negative, 4–5 stars positive, 3 stars dropped (no number, so dropna removes them)
    reviews["y"] = reviews["label"].map({0: 0, 1: 0, 3: 1, 4: 1})


def part(name):
    a, b = split[name]
    rows = reviews.iloc[a:b].dropna(subset=["text", "y"])
    return rows["text"].astype(str).tolist(), rows["y"].astype(int).tolist()


train_x, train_y = part("train")
val_x, val_y = part("validation")  # the test part waits for milestone 5
print(len(train_x), "training and", len(val_x), "validation reviews;", round(sum(train_y) / len(train_y) * 100), "% positive")

## The neutral rule

CAMeLBERT already has three labels: positive, negative and neutral. Your reviews have only two, so neutral never counts as an answer: both runs compare only the positive and negative scores and pick the larger. Fine-tuning keeps the same three-label head and trains it only on positive and negative.

The notebook also reports `neutral_share`, how often neutral would have won. Put it in your notes: it shows how many reviews the model finds unclear.

In [ ]:
import time

import numpy as np
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", DEVICE)  # cpu means no GPU: Runtime → Change runtime type → T4 GPU
tok = AutoTokenizer.from_pretrained(MODEL)


def label_ids(model):
    return {name.lower(): int(i) for i, name in model.config.id2label.items()}  # positive, negative, neutral


def scores(model, texts, batch=64):
    """The model's probability for each of its labels, one row per review."""
    model.to(DEVICE).eval()
    rows = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEVICE)
            rows.append(model(**enc).logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(rows)


def judge(model, texts, y):
    """Scores the validation reviews with the neutral rule: positive against negative only (1 = positive)."""
    ids = label_ids(model)
    p = scores(model, texts)
    pred = (p[:, ids["positive"]] > p[:, ids["negative"]]).astype(int)
    return {
        "accuracy": round(float(accuracy_score(y, pred)), 4),
        "macro_f1": round(float(f1_score(y, pred, average="macro")), 4),
        "recall_negative": round(float(recall_score(y, pred, pos_label=0)), 4),
        "recall_positive": round(float(recall_score(y, pred, pos_label=1)), 4),
        "confusion": confusion_matrix(y, pred, labels=[0, 1]).tolist(),  # rows: really negative, really positive
        "neutral_share": round(float((p.argmax(1) == ids["neutral"]).mean()), 4),
    }


def size_mb(model):
    return round(sum(t.numel() * t.element_size() for t in model.parameters()) / 1e6)

## Run 1: CAMeLBERT as it is

No training: a second baseline, to see what fine-tuning adds.

In [ ]:
import mlflow

mlflow.set_experiment("m3-transformer")
runs = {}

scored = AutoModelForSequenceClassification.from_pretrained(MODEL)
start = time.time()
result = judge(scored, val_x, val_y)
result["seconds"] = round(time.time() - start, 1)
result["size_mb"] = size_mb(scored)
runs["scored"] = result
with mlflow.start_run(run_name="camelbert-scored"):
    mlflow.log_params({"model": MODEL, "fine_tuned": False})
    mlflow.log_metrics({k: v for k, v in result.items() if isinstance(v, (int, float))})
print(result)

## Run 2: fine-tuned with Trainer

About 5 minutes on a T4. If CUDA runs out of memory, set `BATCH = 8`.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

EPOCHS = 2
BATCH = 16
LEARNING_RATE = 2e-5

model = AutoModelForSequenceClassification.from_pretrained(MODEL)  # the same three labels; neutral is never a target
ids = label_ids(model)
train = Dataset.from_dict({"text": train_x, "label": [ids["positive"] if y == 1 else ids["negative"] for y in train_y]})
train = train.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    learning_rate=LEARNING_RATE,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    logging_steps=50,
    report_to="none",  # this notebook logs to MLflow itself, below
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train, data_collator=DataCollatorWithPadding(tok))
start = time.time()
trainer.train()
train_seconds = round(time.time() - start, 1)

start = time.time()
result = judge(trainer.model, val_x, val_y)
result["seconds"] = round(time.time() - start, 1)
result["train_seconds"] = train_seconds
result["size_mb"] = size_mb(trainer.model)
runs["fine_tuned"] = result
with mlflow.start_run(run_name="camelbert-fine-tuned"):
    mlflow.log_params({"model": MODEL, "fine_tuned": True, "epochs": EPOCHS, "batch": BATCH, "learning_rate": LEARNING_RATE})
    mlflow.log_metrics({k: v for k, v in result.items() if isinstance(v, (int, float))})
print(result)
# Milestones 4 and 5 need this model again: trainer.save_model("model") and tok.save_pretrained("model"),
# then download the folder or push it to the Hugging Face Hub.

## Compare the runs

Take a screenshot of the table below and save it as `docs/mlflow.png`. Then decide which run wins, counting time and size too, not only macro F1.

In [ ]:
names = {
    "tags.mlflow.runName": "run",
    "metrics.macro_f1": "macro F1",
    "metrics.accuracy": "accuracy",
    "metrics.neutral_share": "neutral share",
    "metrics.seconds": "seconds to score",
    "metrics.train_seconds": "seconds to train",
    "metrics.size_mb": "size (MB)",
}
table = mlflow.search_runs(experiment_names=["m3-transformer"])
table = table[[c for c in names if c in table.columns]].rename(columns=names)
if os.path.exists("results/baseline.json"):  # your milestone 2 number, the one to beat
    print("TF-IDF baseline macro F1:", json.load(open("results/baseline.json", encoding="utf-8")).get("macro_f1"))
table

In [ ]:
WINNER = max(runs, key=lambda r: runs[r]["macro_f1"])  # change it if time or size matter more to you
WHY = "One sentence: why this run wins, counting macro F1, time and size."

out = {"model": MODEL, "winner": WINNER, "why": WHY, **runs[WINNER], "runs": runs, "test_used": False}
with open("results/transformer.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print("Saved results/transformer.json. Download it from the Files panel and add it to your copy, with docs/mlflow.png.")